# VideoObjectSwap_WanVACE_Colab — video mẫu + ảnh sản phẩm mới → video mới, Wan2.1 VACE trên T4

**Bài toán**: có một video mẫu (sản phẩm A đứng yên trên nền động — ví dụ bảng quảng
cáo có nước chảy tràn) và một ảnh sản phẩm B mới. Muốn ra video mới **giữ nguyên
chuyển động/nền/camera** của video mẫu, chỉ **đổi sản phẩm A → B**, và việc ghép
ảnh (tách nền sản phẩm B, đặt đúng vị trí/ánh sáng) là **tự động**.

Đây chính là logic đứng sau ảnh `last_alternative_..._fd679799....png` (áo sơ mi hồng
ghép vào đúng vị trí chai nước hoa trên tòa nhà) — notebook này tự động hoá bước ghép
đó và thêm bước sinh video.

## Pipeline (2 bước)

1. **Ghép + hoà hợp (preview)**:
   - [rembg](https://github.com/danielgatis/rembg) tách nền ảnh sản phẩm mới (chỉ giữ
     lại sản phẩm, nền trong suốt).
   - Dán thô sản phẩm đã tách nền vào đúng vị trí bbox trên khung hình đầu của video
     mẫu (giữ tỉ lệ, không kéo méo).
   - [FLUX.1 Kontext dev](https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev)
     (bản lượng tử hoá 4-bit, không bị khoá) hoà hợp ánh sáng/bóng đổ/phối cảnh, xoá
     viền cứng của bản dán thô.
   - Đây là bước xem trước rẻ (một ảnh) trước khi tốn GPU cho bước 2. **Có thể tắt**
     (`DUNG_GHEP_ANH_PREVIEW = False`) — bước 2 vẫn chạy được, chỉ là dùng bản dán thô
     (còn nền trắng/viền cứng) thay vì ảnh đã hoà hợp.

2. **Sinh video** — Wan2.1 VACE 14B (bản GGUF, cùng model đã đo trên T4 ở
   `ImageVideoToVideo_WanVACE_Colab.ipynb`), dùng cơ chế **"Masked Video-to-Video
   Editing"** của node `WanVaceToVideo`: khung THẬT của video mẫu làm `control_video`,
   nhưng **đã tự dán sẵn sản phẩm mới vào đúng vùng bbox của MỌI khung** (xem
   `_anh_composite_toan_khung` ở cell 6) trước khi đưa vào VACE — chi tiết cơ chế và lý
   do bắt buộc phải dán trước (không phải chỉ đưa `reference_image`) xem docstring của
   `doi_san_pham_video` ở cell 6 và `CHANGELOG` v3 ở cell 1.

## ⚠️ Đã sửa 2 vòng dựa trên kết quả chạy thật, vẫn CHƯA đo lại

- **v3** (chạy thật, 548s): video ra gần như y hệt bản gốc — không thấy sản phẩm mới.
  Nguyên nhân + cách sửa (tự dán sản phẩm vào `control_video`): xem `CHANGELOG` v3.
- **v4** (dựa trên ảnh kết quả người dùng gửi sau v3): sản phẩm ĐÃ xuất hiện đúng vị
  trí, nhưng là **miếng dán trắng cứng** — vì (a) bước ghép ảnh khi đó (AnyDoor) không
  tách nền/không hoà sáng, và (b) bước sinh video dùng LoRA tốc độ 4-bước/cfg=1 nên
  model gần như không có "sức" hoà trộn. v4 đổi cả hai: AnyDoor → rembg + FLUX Kontext
  (tách nền + hoà sáng thật), và bỏ LoRA tốc độ + tăng lên 25 bước/cfg=5.5 để model đủ
  sức làm thật hoá vùng vừa dán. **Đổi lại: mỗi job sẽ chậm hơn v3 nhiều** (ước lượng
  thô ~5-6 lần thời gian denoise của 548s — CHƯA đo thật). Có thể bật lại
  `dung_lora_toc_do = True` ở `MAC_DINH` (cell 1) để quay về tốc độ v3 nếu 25 bước quá
  lâu trên T4 free tier, đổi lại hoà trộn kém hơn.

Chạy xong (dù đạt hay chưa đạt) nhớ ghi số đo/nhận xét thật vào `CHANGELOG` ở cell 1
như các notebook kia đã làm — đây là cách duy nhất để vòng lặp tinh chỉnh tiếp theo
(nếu cần) biết chính xác đang đứng ở đâu.

## Model dùng

| bước | model | code | license |
|---|---|---|---|
| 1a (tách nền) | rembg (u2net) | [danielgatis/rembg](https://github.com/danielgatis/rembg) | MIT |
| 1b (hoà hợp) | FLUX.1 Kontext dev, bản 4-bit `HighCWu/FLUX.1-Kontext-dev-bnb-hqq-4bit` | [black-forest-labs/FLUX.1-Kontext-dev](https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev) | FLUX.1 [dev] Non-Commercial License — tự kiểm tra trước khi dùng thương mại |
| 2 (video) | Wan2.1 VACE 14B (GGUF) | [ali-vilab/VACE](https://github.com/ali-vilab/VACE) | Apache-2.0 |

Xem chi tiết cân nặng/VRAM ở cell 1 (`MODELS`).

## Yêu cầu GPU

Colab T4 (15 GB VRAM), như đề bài: RAM hệ thống 12.7 GB, RAM GPU 15 GB, ổ đĩa ~112 GB.
Các model cộng lại tải về đĩa ~35 GB (Wan-VACE Q5_K_M ~20 GB + FLUX Kontext 4-bit ~10 GB
+ rembg/RIFE nhỏ) — dư dả trong 112 GB. VRAM dùng TUẦN TỰ (unload giữa các bước), không
cần các model cùng lúc. T4 là kiến trúc Turing (sm_75), không có Tensor Core bf16 —
FLUX Kontext chạy `torch.bfloat16` vẫn hoạt động nhưng chậm hơn GPU Ampere+ (chưa đo).


In [ ]:
# ============================================================================
#  VideoObjectSwap_WanVACE_Colab.ipynb
#  backend = wan_vace_swap   (1 notebook = 1 phuong an, KHONG gop nhieu model 1 file)
#
#  Bai toan: co MOT video mau (san pham A dung yen tren nen dong — vd bang quang
#  cao co nuoc chay), va MOT anh san pham B moi. Muon ra video MOI giu NGUYEN
#  chuyen dong/nen/camera cua video mau, chi doi san pham A -> B.
#
#  KHONG port tu notebook Isi-dev nao — day la thiet ke moi, dung LAI ha tang da
#  co o ImageVideoToVideo_WanVACE_Colab.ipynb (cung fork ComfyUI, cung Wan2.1 VACE
#  GGUF, cung cach tai model). Khac o cho: notebook kia dung VACE kieu "Reference-
#  to-Video" (dwpose/densepose RUT TU video mau + anh nguoi -> BIA lai toan bo
#  khung hinh). Notebook nay dung VACE kieu "Masked Video-to-Video Editing".
#
#  Co che THAT cua WanVaceToVideo (doc thang source comfy_extras/nodes_wan.py —
#  xem CHANGELOG v3 de biet vi sao ban dau hieu SAI): ca 'inactive' (mask=0) va
#  'reactive' (mask=1) deu lay pixel TU CHINH control_video — mask chi chon nhanh
#  nao giu pixel that. KHONG co chuyen 'vung mask=1 tu dong duoc ve lai theo
#  reference_image'. Vi vay pipeline nay TU DAN san pham moi vao MOI khung cua
#  control_video (ham _anh_composite_toan_khung o cell 6) TRUOC KHI dua vao VACE —
#  bien viec cua model tu 'bia ra tu con so' thanh 'lam that/hoa hop mot ban da co
#  hinh dang dung'.
#
#  Pipeline 2 buoc:
#   1) (tuy chon) rembg tach nen anh san pham + dan tho vao khung dau video mau +
#      FLUX.1 Kontext dev (4bit, HighCWu/FLUX.1-Kontext-dev-bnb-hqq-4bit) hoa hop
#      anh sang/bong do/xoa vien -> anh preview.
#   2) Wan2.1 VACE 14B GGUF sinh video, KHONG dung LoRA toc do (lightx2v) — chay du
#      buoc + cfg that de model co du "suc" hoa tron thay vi chi sao chep.
#
#  !!! CHUA CHAY THAT TREN T4 sau lan sua nay !!! Lan chay that dau tien (v3, xem
#  CHANGELOG) cho thay video ra = video goc (khong doi san pham) — da sua ca hai
#  nguyen nhan (dan san pham vao control_video — v3; va gio la bo LoRA toc do +
#  tang buoc/cfg — v4) nhung CHUA DO LAI. Chay xong nho ghi so do that vao CHANGELOG.
# ============================================================================
NOTEBOOK_NAME    = 'VideoObjectSwap_WanVACE_Colab.ipynb'
NOTEBOOK_VERSION = 'v7'        # <-- BUMP moi khi sua notebook + cap nhat CHANGELOG
TEST_CASE        = 'V1-demo-waterfall-billboard'    #@param {type:"string"}
BACKEND          = 'wan_vace_swap'

# Quant cua DiT — chon LUC CAI DAT, doi la phai tai lai. Bang do THAT lay tu
# ImageVideoToVideo_WanVACE_Colab.ipynb (cung file GGUF, NHUNG do voi LoRA toc do +
# 4 buoc — KHONG con dung khi chay day du buoc/cfg nhu o day, xem MAC_DINH):
# Q4_K_M 11,6 GB | Q5_0 12,5 | Q5_K_M 13,0 | Q6_K 14,5 | Q8_0 18,7 (KHONG vua T4).
QUANT_VACE = 'Q5_K_M'    #@param ["Q4_K_M", "Q5_0", "Q5_K_M", "Q6_K"]

# Bat/tat buoc 1 (ghep anh preview: rembg + FLUX Kontext). Tat -> buoc 2 tu dan
# THO anh san pham goc (khong tach nen, khong hoa hop) vao control_video — van
# chay duoc, chi la vien/anh sang se gia hon.
DUNG_GHEP_ANH_PREVIEW = True    #@param {type:"boolean"}

AI_TYPE = 'video_object_swap_wan_vace'
AI_TYPE_ALIAS = ['vace_swap', 'video_doi_san_pham', 'product_swap_video']

MODELS = [
    {
        "name": "Wan2.1 VACE 14B GGUF (masked video-to-video editing), KHONG LoRA toc do",
        "code": "https://github.com/ali-vilab/VACE",
        "weights": "hf:Isi99999/Wan2.1BasedModels (GIONG HET ImageVideoToVideo_WanVACE_Colab.ipynb) "
                   "+ umt5-xxl fp8 + wan_2.1_vae",
        "license": "Apache-2.0 (Wan/VACE)",
        "size": "Q4_K_M 11,6 GB .. Q6_K 14,5 GB (do that o notebook chi em, VOI LoRA/4-buoc — "
                "thoi gian se KHAC nhieu o day vi chay du buoc, xem MAC_DINH) + umt5 6,7 GB + VAE"
    },
    {
        "name": "rembg (u2net) — tach nen anh san pham (BUOC 1a)",
        "code": "https://github.com/danielgatis/rembg",
        "weights": "tu tai model u2net (~176 MB) o lan dung dau",
        "license": "MIT",
        "size": "~200 MB"
    },
    {
        "name": "FLUX.1 Kontext dev, ban luong tu hoa 4-bit (BUOC 1b: hoa hop anh preview)",
        "code": "https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev",
        "weights": "hf:HighCWu/FLUX.1-Kontext-dev-bnb-hqq-4bit (KHONG bi gate, khong can chap nhan "
                   "giay phep tren HF — ban cong dong luong tu hoa san: DiT bnb-nf4-4bit + T5 hqq-4bit)",
        "license": "FLUX.1 [dev] Non-Commercial License (BFL) — KIEM TRA truoc khi dung thuong mai",
        "size": "~10 GB"
    }
]

BACKEND_PARAMS = ('video_url', 'anh_san_pham_url', 'bbox_san_pham_cu', 'mo_ta_san_pham_moi',
                  'negative_prompt', 'so_frame', 'moi_khung_thu', 'khung_moi_giay', 'rong', 'cao',
                  'steps', 'guidance', 'seed', 'fps_ra', 'mo_rong_mask', 'dung_ghep_anh_preview',
                  'dung_lora_toc_do')

# Bo mac dinh — CHUA DO THAT tren T4, chi la diem xuat phat hop ly (xem markdown dau file).
MAC_DINH = {
    "so_frame": 41,                # 4n+1 nhu VACE doi hoi (xem _khung_4n1)
    "moi_khung_thu": 0,            # 0 = tu tinh theo khung_moi_giay
    "khung_moi_giay": 8.0,         # lay moi khung thu N cua video mau de vua ngan sach GPU
    "fps_ra": 24,
    "ngan_sach_pixel": 458752,     # 512x896, giong ImageVideoToVideo_WanVACE_Colab.ipynb
    # KHONG con dung LoRA toc do (lightx2v) theo mac dinh -> can THEM buoc VA cfg THAT de
    # model co suc hoa tron/lam that vung da dan san pham moi, thay vi chi sao chep nhu o
    # v3 (4 buoc, cfg 1.0 danh rieng cho LoRA chung cat). 20-25 buoc, cfg 5-6 la khoang
    # PHO BIEN cong dong dung cho Wan KHONG LoRA — CHUA DO LAI tren T4, se cham hon
    # NHIEU so voi 4 buoc (uoc luong: gap ~5-6 lan thoi gian denoise cua v3).
    "steps": 25,
    "guidance": 5.5,
    "sampler": "euler_ancestral",
    "scheduler": "simple",
    # shift 8.0 o v3 la tuned CHO LoRA lightx2v; bo LoRA thi 8.0 co the qua cao. 5.0 la
    # gia tri pho bien hon cho Wan2.1 chay chuan (KHONG kiem chung rieng cho video nay).
    "shift": 5.0,
    "dung_lora_toc_do": False,     # True = quay lai kieu v3 (4 buoc, cfg 1, lightx2v) — NHANH nhung
    "lora": "lightx2v",            # it "suc" hoa tron hon; doi dong nay KHONG can chay lai cell 2.
    "lightx2v": 1.0,
    # Mask vung san pham CU trong khung hinh mau: MO RONG them bao nhieu px (sau khi da
    # resize ve rong x cao) de VACE co doi cho san pham moi khac hinh dang san pham cu
    # (vd chai tron -> ao so mi vuong) ma khong bi mask cat cut.
    "mo_rong_mask": 24,
    "mo_ta_san_pham_moi": "the exact same billboard advertisement scene, same building, same camera "
                          "angle, same cascading water effect, same lighting — only the product itself "
                          "is different, photorealistic, high quality. Completely replace the original "
                          "product; no trace of its old shape, color or texture should remain visible. "
                          "Keep the new product at its given size and position, do not shrink it.",
    "negative_prompt": "bad quality, blurry, warped building, extra objects, flickering, deformed logo, "
                       "text artifacts, low resolution",
    # Prompt cho FLUX Kontext o buoc 1b (hoa hop anh preview) — xem cell 5.
    "prompt_hoa_hop": "Blend the newly placed product naturally into the scene: add realistic "
                      "lighting, reflections, shadows and perspective consistent with the "
                      "surrounding billboard/building, remove any white or flat background around "
                      "the pasted object, keep everything else in the image exactly the same. "
                      "Do NOT resize, shrink, enlarge or move the pasted product — keep its exact "
                      "size, position and proportions unchanged. Completely remove and replace the "
                      "original product underneath; no trace of its old shape or color should remain.",
    "flux_guidance": 2.5,
    "flux_steps": 28,
}

CHANGELOG = {
    "v7": "DO THAT tren T4 voi v6 (sau khi ti le san pham moi da dung — nguoi dung xac nhan "
          "'san pham da theo ti le roi'): san pham CU (chai) VAN con lo ra hai ben canh san "
          "pham moi (ao so mi hep hon nhieu so voi be ngang chai). NGUYEN NHAN: o v6 moi chi "
          "sua MASK (bbox_mask = hop cua bbox goc + vi tri dat) cho VACE, nhung ANH THUC TE dua "
          "cho VACE lam 'reactive' (control_video da dan) — tuc anh_ref/tho — CHUA duoc xoa "
          "sach trong vung bbox_mask do; ham dan san pham (_dan_tho_vao_khung, "
          "_anh_composite_toan_khung) van CHI dan san pham moi vao dung hinh chu nhat cua no "
          "(vi_tri_dat, hep hon bbox_mask), phan con lai cua bbox_mask (hai canh, vi san pham "
          "moi hep hon bbox chai cu) VAN la pixel san pham CU tu canvas nen. Mask co du rong "
          "(tu v6) khong quan trong bang NOI DUNG pixel ben trong no — day moi la bai hoc dung: "
          "phai XOA SACH TRUOC roi moi DAN, khong chi 'mo mask cho rong ra'. "
          "SUA: ca hai ham dan san pham gio TO MAU XAM TRUNG TINH (127,127,127) cho TOAN BO "
          "bbox_mask (+ mo_rong_mask) TRUOC KHI dan san pham moi len tren — 127 la quy uoc "
          "CHINH THUC cua VACE cho 'vung can sinh lai, khong co gi o day' (doc tu docs/VACE.md "
          "cua WanGP, muc 'VACE Control Video And Mask Format': 'Grey, value 127, means the "
          "area is replaced by new content'). Nho vay KHONG con pixel san pham cu nao lot qua "
          "trong toan bo vung mask, bat ke san pham moi hep/rong hon bbox bao nhieu. "
          "CHUA DO LAI tren T4 sau sua nay.",
    "v6": "DO THAT tren T4 voi v5 (sau khi sua bbox trum het chai cu theo huong dan): video ra "
          "van KHONG xoa duoc chai cu, VA ao moi bi thu nho 'lot thom' giua khung — ca hai TRIEU "
          "CHUNG NAY co CHUNG mot nguyen nhan hinh hoc. Ham dan san pham cu (_dan_tho_vao_khung / "
          "_anh_composite_toan_khung) dung kieu 'fit-contain vao TRON bbox': giu ti le san pham "
          "MOI nhung neu ti le bbox nguoi dung khoanh (rong/cao) LECH voi ti le tu nhien cua san "
          "pham moi thi con KHOANG HO o hai canh hoac tren-duoi bbox — vung ho do KHONG duoc dan "
          "de len, nen la` pixel SAN PHAM CU (tu khung nen o duoi) lo qua. Vung ho nay VAN nam "
          "trong mask (mask=1, 'reactive') nhung noi dung dua cho VACE tai do lai la san pham CU, "
          "nen model 'hoc' duoc la phai giu nguyen no — dung la nguyen nhan chai khong bi xoa. "
          "Dong thoi san pham moi (dan theo dung ti le, nen NHO hon bbox theo 1 chieu) nhin 'lot "
          "thom' vi bbox nguoi dung khoanh (theo hinh dang chai) lon hon nhieu so voi khung hinh "
          "chu nhat that su cua ao so mi. "
          "SUA: them ham dat_san_pham_theo_ti_le() (cell 3) — LAP DAY dung chieu cao bbox (tin "
          "chieu cao nguoi dung khoanh la dung), tinh chieu rong theo DUNG ti le tu nhien cua san "
          "pham moi (khong bop meo, khong con khoang ho theo chieu doc), can giua theo chieu "
          "ngang cua bbox. Mask dung cho VACE gio la HOP (union) cua bbox goc (che het san pham "
          "cu, du hep/rong hon san pham moi) va vi tri dat san pham moi thuc te (phong ra neu no "
          "rong hon bbox) — dam bao ca hai deu nam TRON trong vung 'reactive'. Cap nhat prompt "
          "(mo_ta_san_pham_moi + prompt_hoa_hop) noi ro 'khong duoc thu nho/di chuyen san pham, "
          "xoa het dau vet san pham cu' de FLUX Kontext (buoc 1b) khong tu y ve lai san pham nho "
          "hon 'cho tu nhien'. CHUA DO LAI tren T4 sau sua nay.",
    "v5": "DO THAT tren T4 voi v4: video ra van la mieng dan CO NGUYEN NEN TRANG + san pham cu "
          "van lo ra tren/duoi mieng dan. Nguyen nhan (1): ham tao_anh_preview() cu boc CA rembg "
          "LAN FLUX Kontext trong CUNG mot try/except — chi can 1 trong 2 loi la ham tra ve None "
          "va cell 6 lui thang ve dan NGUYEN anh goc (con nen), nen KHONG THE biet buoc nao that "
          "su loi tu log. Nguyen nhan (2), theo doi bbox: khong du che het chieu cao vat the cu. "
          "SUA (1): tach rieng 2 try/except — rembg loi thi bao ro 'TACH NEN (rembg) loi ...'; "
          "rembg on nhung FLUX Kontext loi thi VAN dung ban DA TACH NEN (chi la chua hoa anh "
          "sang) lam anh preview thay vi bo het ve None — nghia la CHI CAN rembg chay duoc la nen "
          "trang da bien mat, khong con phu thuoc FLUX Kontext (12B, de loi/timeout tren T4) cho "
          "viec do. Doi chieu code rembg da chay THAT tren mot notebook khac cua nguoi dung "
          "(video_object_swap_colab.ipynb, dung AnyDoor+SAM2+RAFT) de dam bao dung cach goi: "
          "new_session('u2net') tai 1 lan dung lai, va ghim ca 'rembg' LAN 'onnxruntime' cung "
          "dong pip voi numpy ghim san (truoc chi ghim rembg, thieu onnxruntime tuong minh). "
          "Nguyen nhan (2) — bbox chua du lon — la tham so nguoi dung tu chon, CHUA sua trong "
          "code; nhac lai o markdown: dung anh luoi toa do o cell 7 de kiem tra bbox trum HET "
          "vat the cu (ke ca phan chu/nhan) truoc khi chay that. CHUA DO LAI tren T4 sau sua nay.",
    "v4": "Doi ca hai diem yeu ma nguoi dung phat hien tu anh xuat thuc te (mieng dan trang cung, "
          "khong hoa vao canh): "
          "(1) BUOC GHEP ANH: bo AnyDoor (chi dan tho, khong tach nen/khong hoa sang — do do la "
          "nguyen nhan true tiep cua 'mieng dan trang'), thay bang rembg (tach nen sach) + FLUX.1 "
          "Kontext dev 4-bit (hoa hop anh sang/bong do/xoa vien). Ca hai deu chay TRONG moi truong "
          "chinh (khong can venv/subprocess nhu AnyDoor) — rembg khong xung dot gi, con "
          "HighCWu/FLUX.1-Kontext-dev-bnb-hqq-4bit la ban PRE-QUANTIZED cong dong, khong bi gate "
          "(khac ban goc black-forest-labs can chap nhan giay phep + tai token HF), nen tai thang "
          "duoc, ~10 GB thay vi ~24 GB ban goc. "
          "(2) BUOC SINH VIDEO: bo LoRA toc do lightx2v (v3 dung 4 buoc/cfg=1 CHUYEN DE cho LoRA "
          "nay — day cung la mot phan ly do model 'khong du suc' lam that vung da dan o v3), tang "
          "len 25 buoc + cfg 5.5, ha shift tu 8.0 (tuned rieng cho LoRA) xuong 5.0 (gia tri pho "
          "bien hon cho Wan2.1 chay chuan). Gia phai tra: MOI JOB SE CHAM HON NHIEU (uoc luong tho: "
          "denoise ti le thuan so buoc, tuc ~5-6x thoi gian denoise cua v3's 548s tong — CAN DO "
          "THAT, chua chay lai). Giu lai cong tac 'dung_lora_toc_do' de quay ve kieu v3 (nhanh, it "
          "hoa tron) neu 25 buoc qua lau tren T4 free tier. CHUA DO LAI toan bo sau sua nay.",
    "v3": "DO THAT tren T4 lan dau (job V1-demo-waterfall-billboard, 41 khung, 548s): chay "
          "het pipeline khong loi, NHUNG video ra gan nhu Y HET video mau — khong thay san "
          "pham moi dau ca. Nguyen nhan (doc thang source comfy_extras/nodes_wan.py, "
          "WanVaceToVideo.encode, KHONG con suy tu tai lieu ban tom tat nua): "
          "'inactive = control_video*(1-mask)' VA 'reactive = control_video*mask' — CA HAI "
          "nhanh deu lay pixel TU CHINH control_video, mask chi chon nhanh nao giu pixel "
          "that/nhanh nao thay xam. Doan v2 sai la tuong 'vung mask=1 se duoc ve lai theo "
          "reference_image' — THAT RA reference_image chi la mot khung rieng noi VAO DAU "
          "chuoi latent (goi y toan cuc yeu), KHONG ep pixel vao vung mask. Cong voi "
          "guidance=1.0 (bat buoc cho LoRA lightx2v 4-buoc, khong doi duoc) lam prompt mo ta "
          "san pham moi gan nhu vo tac dung, nen vung 'reactive' chi la anh SAN PHAM CU ve "
          "lai y nguyen. SUA: them ham _anh_composite_toan_khung() — TU DAN san pham moi "
          "(tu anh_preview neu co, khong thi tu dan tho anh_san_pham fit-contain vao dung "
          "bbox tren nen khung dau video_mau) vao MOI khung cua control_video TRUOC KHI dua "
          "vao WanVaceToVideo. Nho vay vung 'reactive' gio la SAN PHAM MOI (tho), model chi "
          "con phai 'lam that/hoa hop anh sang-chuyen dong nuoc' thay vi bia tu con so. Ket "
          "qua thuc te (anh nguoi dung gui): san pham moi DA XUAT HIEN dung vi tri, nhung la "
          "MIENG DAN TRANG CUNG (chua tach nen, chua hoa sang) — dan den sua tiep o v4.",
    "v2": "Sua cell 4 (moi truong AnyDoor) sau 3 lan chay that tren T4 deu vo o tang HA TANG, "
          "khong phai o logic AnyDoor: (1) apt-get install ... python3-venv chet CA CUM lenh vi ten "
          "goi apt doi theo ban Python cua tung anh Colab (vd python3.13-venv) — huy luon aria2/ffmpeg "
          "di kem; (2) chuyen sang Miniconda thi conda create vo vi kenh 'defaults' cua Anaconda gio "
          "doi chap nhan Dieu khoan dich vu, khong lam duoc trong Colab khong tuong tac; (3) chuyen "
          "kenh sang conda-forge thi qua duoc buoc tao moi truong, nhung pip install -r "
          "requirements.txt (ghim Python 3.8/2023) van loi tren Python 3.10 moi tao vi vai goi "
          "2020-era (vd opencv-contrib-python 4.3.0.36) khong co wheel. "
          "QUYET DINH: bo huong 'moi truong rieng' hoan toan. Doc het cldm/*.py + ldm/*.py cua chinh "
          "AnyDoor xac nhan day chi la code nn.Module/attention/DDIM THUAN PyTorch, moi cho dung "
          "xformers (cldm/hack.py, dinov2/layers/attention.py + swiglu_ffn.py, ldm/modules/attention.py) "
          "deu co try/except ImportError du phong ve attention thuong — an toan chay THANG trong moi "
          "truong chinh (torch 2.6.0 cua ComfyUI o cell 2), chi can cai them pytorch_lightning<2 "
          "(<2 vi ldm/models/diffusion/ddpm.py dung duong dan pytorch_lightning.utilities.distributed."
          "rank_zero_only, bi doi cho tu ban 2.0), omegaconf, open_clip_torch. Van goi qua subprocess "
          "rieng (khong import chung tien trinh) nen VRAM van giai phong sach. Phat hien them 1 loi "
          "AN chua kip chay toi: ldm/modules/encoders/modules.py lam 'import hubconf' o CAP MODULE — "
          "chi resolve duoc neu thu muc AnyDoor/dinov2 (ban sao chep dinov2 kem hubconf.py) nam trong "
          "sys.path; da them sys.path.insert() dau anydoor_infer.py de phong truoc. (AnyDoor DA BI BO "
          "hoan toan o v4, ghi lai day chi de nho lai duong di.)",
    "v1": "Ban dau. Thiet ke dua tren: (1) tai lieu chinh thuc WanVaceToVideo cua ComfyUI "
          "(docs.comfy.org) ve co che control_masks 'reactive'/'inactive'; (2) bao cao cong dong "
          "9/2026 ve quy trinh 'object swap' voi WAN 2.2 VACE (video driver + anh dau da sua bang "
          "Nano Banana + mask vung can doi); (3) ha tang da do that o "
          "ImageVideoToVideo_WanVACE_Colab.ipynb (cung DiT/quant/LoRA). CHUA TU DO THOI GIAN/VRAM "
          "that cua rieng notebook nay tren T4 — buoc dau tien khi chay that la GHI LAI so do vao day."
}

import shutil, subprocess


def _sh(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
    except Exception:
        return ''


print(f'=== {NOTEBOOK_NAME} {NOTEBOOK_VERSION} | backend={BACKEND} | case={TEST_CASE} ===')
for m in MODELS:
    print(f"    model: {m['name']}  [{m['license']}]")
    print(f"           {m['weights']}")
print('GPU:', _sh(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
      or '!!! CHUA BAT GPU: Runtime > Change runtime type > GPU (T4)')
print('DUNG_GHEP_ANH_PREVIEW =', DUNG_GHEP_ANH_PREVIEW,
      '(tat neu cell 4/5 loi — van chay duoc buoc 2 voi anh san pham goc dan tho)')
print('dung_lora_toc_do =', MAC_DINH['dung_lora_toc_do'],
      f'(steps={MAC_DINH["steps"]}, cfg={MAC_DINH["guidance"]} — CHAM hon v3 nhieu, xem CHANGELOG v4)')
print('ffmpeg:', shutil.which('ffmpeg') or 'THIEU (cell cai dat se cai)')


In [ ]:
import os, shutil, subprocess, sys, time

COMFY = '/content/ComfyUI'
_T0 = time.time()


def _run(cmd, cwd=None, check=True, quiet=True):
    """Chay lenh, in dong lenh truoc de log Colab doc duoc dang o buoc nao."""
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str),
                       capture_output=quiet, text=True)
    if check and r.returncode != 0:
        print(r.stdout[-3000:] if r.stdout else '')
        print(r.stderr[-3000:] if r.stderr else '')
        duoi = (r.stderr or r.stdout or '').strip()[-1200:]
        raise RuntimeError(f'lenh loi ({r.returncode}): {cmd}' + chr(10) + '--- stderr ---' + chr(10) + duoi)
    return r


def _pip(*pk):
    _run([sys.executable, '-m', 'pip', 'install', '-q', *pk])


def _clone(url, dst, branch=None):
    if os.path.isdir(os.path.join(dst, '.git')):
        print(f'   da co {dst}, bo qua clone')
        return
    cmd = ['git', 'clone', '-q', '--depth', '1']
    if branch:
        cmd += ['--branch', branch]
    _run(cmd + [url, dst])


def _tai_requests(url, dst):
    """Du phong khi khong co aria2c: tai 1 luong bang requests (cham hon nhung KHONG
    phu thuoc apt). Dung stream + chunk de khong nghen RAM voi file 10+ GB."""
    import requests
    r = requests.get(url, stream=True, timeout=600, headers={'User-Agent': 'Mozilla/5.0'})
    r.raise_for_status()
    with open(dst, 'wb') as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    return dst


def _aria(url, thu_muc, ten=None):
    """Tai bang aria2c 16 luong. Khong co aria2c (vd apt-get o cell nay bi loi) thi lui
    ve requests 1 luong — cham hon nhung KHONG lam ca notebook chet. Co file roi thi bo qua."""
    ten = ten or url.split('/')[-1].split('?')[0]
    os.makedirs(thu_muc, exist_ok=True)
    d = os.path.join(thu_muc, ten)
    if os.path.exists(d) and os.path.getsize(d) > 1e6:
        print(f'   da co {ten} ({os.path.getsize(d) / 1e9:.2f} GB)')
        return d
    t0 = time.time()
    if shutil.which('aria2c'):
        _run(['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M',
              '--summary-interval=0', '-d', thu_muc, '-o', ten, url])
    else:
        print(f'   !!! aria2c KHONG co trong PATH -> tai {ten} bang requests (1 luong, cham hon)')
        _tai_requests(url, d)
    print(f'   {ten}: {os.path.getsize(d) / 1e9:.2f} GB trong {time.time() - t0:.0f}s')
    return d


# check=False truoc day AN LUON loi that (vd "Unable to locate package") roi chet 40 dong
# sau o _aria voi thong bao khong lien quan ("No such file: aria2c"). Gio neu apt-get loi
# thi IN RO loi va apt-get update roi thu lai 1 lan, khong am tham nuot loi nua.
_apt = _run(['apt-get', '-y', 'install', '-qq', 'aria2', 'ffmpeg'], check=False)
if _apt.returncode != 0:
    print('   !!! apt-get install loi lan 1, in log va thu apt-get update roi cai lai:')
    print((_apt.stdout or '')[-1500:]); print((_apt.stderr or '')[-1500:])
    _run(['apt-get', 'update', '-qq'], check=False)
    _apt = _run(['apt-get', '-y', 'install', '-qq', 'aria2', 'ffmpeg'], check=False)
    if _apt.returncode != 0:
        print('   !!! apt-get van loi lan 2 — se DUNG DUOC (fallback ffmpeg co san tren Colab,'
              ' _aria lui ve requests) nhung cham hon, va RIFE inference_video.py can ffmpeg:')
        print((_apt.stdout or '')[-1500:]); print((_apt.stderr or '')[-1500:])
if not shutil.which('ffmpeg'):
    raise RuntimeError('khong co ffmpeg sau ca 2 lan apt-get — kiem tra log o tren, hoac chay '
                       "'!apt-get update && !apt-get install -y ffmpeg' bang tay o mot cell rieng")

# Torch GHIM 2.6.0 — ComfyUI/Wan-VACE can dung ban nay. rembg/FLUX Kontext (buoc 1, cell
# 4/5) chay CHUNG moi truong nay, khong can venv rieng.
_pip('torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0')

_pip('torchsde', 'av', 'diffusers', 'transformers', 'xformers==0.0.29.post2', 'accelerate',
     'tqdm', 'color-matcher', 'onnxruntime', 'onnxruntime-gpu', 'einops', 'spandrel',
     'opencv-python', 'imageio[ffmpeg]', 'psutil', 'requests')

# Cung fork ComfyUI + ComfyUI_GGUF nhu ImageVideoToVideo_WanVACE_Colab.ipynb (da do that
# tren T4). KHONG can comfyui_controlnet_aux o day: notebook nay dung khung THAT cua
# video mau lam control_video (khong qua dwpose/depth/canny) — xem cell 1.
_clone('https://github.com/Isi-dev/ComfyUI', COMFY, branch='ComfyUI_v0.3.36')
_clone('https://github.com/Isi-dev/ComfyUI_GGUF.git', f'{COMFY}/custom_nodes/ComfyUI_GGUF')
_pip('aiohttp', 'yarl', 'pyyaml', 'scipy', 'tqdm', 'psutil', 'safetensors', 'sentencepiece',
     'tokenizers', 'kornia', 'soundfile', 'pydantic', 'pydantic-settings')
_pip('-r', f'{COMFY}/custom_nodes/ComfyUI_GGUF/requirements.txt')

# RIFE: model chi sinh ~so_frame khung o khung_moi_giay fps (ngan sach GPU), RIFE noi
# suy len fps_ra de video muot / dung do dai thoi gian that.
RIFE = '/content/Practical-RIFE'
_clone('https://github.com/Isi-dev/Practical-RIFE', RIFE)
_pip('git+https://github.com/rk-exxec/scikit-video.git@numpy_deprecation')
os.makedirs(f'{RIFE}/train_log', exist_ok=True)
for _f in ('IFNet_HDv3.py', 'RIFE_HDv3.py', 'refine.py', 'flownet.pkl'):
    if not os.path.exists(f'{RIFE}/train_log/{_f}'):
        _run(f'curl -sL -o {RIFE}/train_log/{_f} https://huggingface.co/Isi99999/'
             f'Frame_Interpolation_Models/resolve/main/4.25/train_log/{_f}')

# Quant lay tu QUANT_VACE o cell 1. Repo goc dat ten file theo HAI kieu khac nhau tuy
# quant — viet thang ra bang chu (giong ImageVideoToVideo_WanVACE_Colab.ipynb).
QUANT = QUANT_VACE
_TEN_GGUF = {
    'Q4_K_M': 'Wan2.1_14B_VACE-Q4_K_M.gguf',
    'Q5_0':   'Wan2.1_14B_VACE-Q5_0.gguf',
    'Q5_K_M': 'Wan2.1-VACE-14B-Q5_K_M.gguf',
    'Q6_K':   'Wan2.1-VACE-14B-Q6_K.gguf',
}
if QUANT not in _TEN_GGUF:
    raise RuntimeError(f'QUANT_VACE={QUANT!r} khong co trong bang; chon mot trong {list(_TEN_GGUF)}')
DM = f'{COMFY}/models/diffusion_models'
DIT = _aria(f'https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/{_TEN_GGUF[QUANT]}', DM)
LORA_LIGHTX2V = _aria(
    'https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/'
    'lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank32_bf16.safetensors', f'{COMFY}/models/loras')
TE = _aria('https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/'
           'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', f'{COMFY}/models/text_encoders')
VAE = _aria('https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/'
            'split_files/vae/wan_2.1_vae.safetensors', f'{COMFY}/models/vae')

print(f'>>> CAI DAT XONG NHANH CHINH (ComfyUI + Wan-VACE GGUF) trong {time.time() - _T0:.0f}s.')


In [ ]:
import json, os, re, shutil, subprocess, time

import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageOps

WRK_DIR = '/content/work'
OUT_DIR = '/content/output'
for d in (WRK_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

LAST_STATS = {}


def _slug(s, default='case'):
    s = re.sub(r'[^A-Za-z0-9._-]+', '-', (s or '').strip()).strip('-.')
    return (s or default)[:48]


def _moc(ten):
    """Moc bo nho (RAM/VRAM) tai tung buoc. T4 hay chet o RAM 12,7 GB chu khong phai
    VRAM, nen phai thay duoc RAM tang o buoc nao."""
    try:
        import psutil
        ram = round(psutil.Process(os.getpid()).memory_info().rss / 1024 ** 3, 2)
    except Exception:
        ram = None
    try:
        vram = round(torch.cuda.memory_allocated() / 1024 ** 3, 2) if torch.cuda.is_available() else None
    except Exception:
        vram = None
    LAST_STATS.setdefault('_bo_nho', {})[ten] = {'ram_gb': ram, 'vram_gb': vram, 't': round(time.time(), 1)}
    print(f'   [moc] {ten}: RAM={ram} GB, VRAM={vram} GB')


def probe_video(path):
    out = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
         'stream=width,height,r_frame_rate,nb_frames', '-of', 'json', path],
        capture_output=True, text=True, check=True).stdout
    st = json.loads(out)['streams'][0]
    num, den = (st['r_frame_rate'].split('/') + ['1'])[:2]
    fps = float(num) / float(den or 1)
    return int(st['width']), int(st['height']), fps, int(st.get('nb_frames') or 0)


def encode_video(frames_dir, fps, out_path, tag=''):
    """PNG -> mp4 h264 yuv420p. Metadata comment mang du notebook + version + backend + case."""
    comment = (f'notebook={NOTEBOOK_NAME} version={NOTEBOOK_VERSION} backend={BACKEND} {tag}').strip()
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-framerate', f'{fps:.6f}', '-start_number', '0',
                    '-i', os.path.join(frames_dir, '%06d.png'),
                    '-c:v', 'libx264', '-preset', 'medium', '-crf', '18', '-pix_fmt', 'yuv420p',
                    '-metadata', f'comment={comment}', out_path], check=True)
    return out_path


def reencode_video(src, out_path, tag=''):
    comment = (f'notebook={NOTEBOOK_NAME} version={NOTEBOOK_VERSION} backend={BACKEND} {tag}').strip()
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', src, '-c:v', 'libx264', '-preset', 'medium',
                    '-crf', '18', '-pix_fmt', 'yuv420p', '-metadata', f'comment={comment}', out_path],
                   check=True)
    return out_path


def save_frames_u8(arr, out_dir):
    """arr: (N,H,W,3) uint8 -> 000000.png ..."""
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    for i in range(arr.shape[0]):
        Image.fromarray(arr[i]).save(os.path.join(out_dir, f'{i:06d}.png'))
    return out_dir


def doc_anh_rgb(path):
    """PNG RGBA -> ep len nen TRANG roi RGB. .convert('RGB') thang se bien vung trong
    suot thanh mau rac."""
    im = Image.open(path)
    im = ImageOps.exif_transpose(im)
    if im.mode in ('RGBA', 'LA') or (im.mode == 'P' and 'transparency' in im.info):
        im = im.convert('RGBA')
        bg = Image.new('RGBA', im.size, (255, 255, 255, 255))
        im = Image.alpha_composite(bg, im)
    return im.convert('RGB')


def _kich_thuoc_theo_anh(w0, h0, ngan_sach, boi=16, toi_thieu=256):
    """Giu ti le anh vao, tong pixel ~= ngan_sach, moi canh chia het cho `boi`."""
    ti_le = w0 / max(1, h0)
    h = (ngan_sach / ti_le) ** 0.5
    w = h * ti_le
    w = max(toi_thieu, int(round(w / boi)) * boi)
    h = max(toi_thieu, int(round(h / boi)) * boi)
    return w, h


def _khung_4n1(n, toi_thieu=5):
    """Wan: so khung phai la 4n+1. Lam tron XUONG va bao ro."""
    n = max(toi_thieu, int(n))
    return (n - 1) // 4 * 4 + 1


def doc_khung_dau(video):
    """Khung dau tien cua video mau, dang PIL RGB — dung de: (a) ghep anh preview
    (cell 5), (b) nguoi dung nhin toa do de dien bbox_san_pham_cu."""
    cap = cv2.VideoCapture(video)
    ok, f = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f'khong doc duoc khung dau cua video: {video}')
    return Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))


def doc_khung_video(video, moi_thu, so_khung):
    """Doc video mau, lay moi khung thu `moi_thu`, toi da `so_khung` -> tensor (N,H,W,3) [0,1]."""
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    ra, i = [], 0
    while len(ra) < so_khung:
        ok, f = cap.read()
        if not ok:
            break
        if i % moi_thu == 0:
            ra.append(torch.from_numpy(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)).float() / 255.0)
        i += 1
    cap.release()
    if not ra:
        raise ValueError('video mau khong doc duoc khung nao')
    return torch.stack(ra, 0), fps


def ve_bbox_len_anh(anh_pil, bbox):
    """Ve khung do quanh bbox len ban sao cua anh — de nguoi dung xac nhan truoc khi chay.
    bbox: (x1,y1,x2,y2) theo pixel THAT cua anh_pil."""
    im = anh_pil.copy()
    d = ImageDraw.Draw(im)
    d.rectangle(bbox, outline=(255, 0, 0), width=max(2, im.width // 200))
    return im


def mat_na_tu_bbox(w, h, bbox, mo_rong=0):
    """Mask nhi phan (H,W) float32 [0,1]: 1 trong bbox (da MO RONG them `mo_rong` px moi
    canh), 0 ngoai. Mo rong (khong phai giam) vi san pham MOI co the khac hinh dang san
    pham CU (vd chai tron -> ao so mi vuong) — mask sat qua se cat cut san pham moi.
    bbox theo pixel cua khung KICH THUOC (w,h) — resize bbox truoc khi goi ham nay neu
    bbox lay tu anh kich thuoc khac (xem `doi_ti_le_bbox`)."""
    x1, y1, x2, y2 = bbox
    x1, x2 = sorted((max(0, x1 - mo_rong), min(w, x2 + mo_rong)))
    y1, y2 = sorted((max(0, y1 - mo_rong), min(h, y2 + mo_rong)))
    m = np.zeros((h, w), dtype=np.float32)
    m[int(y1):int(y2), int(x1):int(x2)] = 1.0
    return m


def doi_ti_le_bbox(bbox, w_goc, h_goc, w_moi, h_moi):
    """bbox do o anh (w_goc,h_goc) -> quy doi ve anh da resize (w_moi,h_moi)."""
    sx, sy = w_moi / w_goc, h_moi / h_goc
    x1, y1, x2, y2 = bbox
    return (x1 * sx, y1 * sy, x2 * sx, y2 * sy)


def dat_san_pham_theo_ti_le(bbox_cu, ti_le_rong_cao_san_pham_moi):
    """Cho bbox san pham CU (x1,y1,x2,y2) + ti le rong/cao TU NHIEN cua san pham MOI ->
    tra ve (vi_tri_dat, bbox_mask).

    Truoc day: dan san pham moi kieu "fit-contain" VAO TRON bbox cu (giu ti le nhung co
    the ho khong gian hai ben/tren-duoi neu ti le bbox va ti le san pham lech nhau) — HAI
    HAU QUA: (1) phan ho do do la pixel san pham CU con lo ra (chua bi dan de len), VACE
    doc dung cai do lam "reactive" nen ve lai dung san pham cu; (2) san pham moi trong "lot
    thom" giua mot bbox lon hon no nhieu.

    Sua: LAP DAY dung chieu cao cua bbox_cu (gia dinh nguoi dung khoanh dung chieu cao
    that cua san pham cu), tinh chieu rong theo DUNG ti le tu nhien cua san pham moi
    (khong bop meo), can giua theo truc ngang cua bbox_cu — khong con khoang ho theo
    chieu doc. bbox_mask = hop (union) cua bbox_cu va vi_tri_dat, de mask luon che HET ca
    vung san pham cu (theo chieu ngang, phong khi san pham moi hep hon) LAN vung san pham
    moi (phong khi no rong hon bbox_cu)."""
    x1, y1, x2, y2 = bbox_cu
    h = y2 - y1
    w = h * ti_le_rong_cao_san_pham_moi
    cx = (x1 + x2) / 2
    vi_tri_dat = (cx - w / 2, y1, cx + w / 2, y2)
    bbox_mask = (min(x1, vi_tri_dat[0]), min(y1, vi_tri_dat[1]),
                max(x2, vi_tri_dat[2]), max(y2, vi_tri_dat[3]))
    return vi_tri_dat, bbox_mask


print('   [helper] da nap ham dung chung (video io, mask tu bbox, mask tu anh san pham)')


In [ ]:
import time

FLUX_KONTEXT_REPO = 'HighCWu/FLUX.1-Kontext-dev-bnb-hqq-4bit'

if not DUNG_GHEP_ANH_PREVIEW:
    print('   [ghep_anh] DUNG_GHEP_ANH_PREVIEW=False -> bo qua cai dat rembg/FLUX Kontext. '
          'Buoc 2 se tu dan THO anh san pham goc (khong tach nen, khong hoa hop).')
else:
    _t0 = time.time()

    # Ca rembg va FLUX Kontext chay TRONG moi truong chinh (torch 2.6.0) — khong can venv
    # rieng nhu AnyDoor tung can: rembg la thu vien onnx thuan (khong dung torch), va ban
    # FLUX Kontext duoi day la goi PRE-QUANTIZED (4-bit) tai san tu HF, tuong thich diffusers
    # hien tai, khong doi phien ban torch cu the.
    #
    # GHIM numpy VE DUNG BAN DANG CHAY: pip giai quyet phu thuoc cua diffusers/transformers/
    # bitsandbytes/hqq co the AM THAM doi phien ban numpy (len hoac XUONG) so voi ban torch/
    # opencv/ComfyUI o cell 2 da bien dich cung — hai ban numpy python-code vs C-extension
    # lech nhau se loi kieu 'module numpy._core._multiarray_umath has no attribute ...' o
    # BAT KY lan import numpy sau do (kho doan truoc no no o dau, vi numpy duoc import khap
    # noi). Ghim tuong minh trong CUNG mot lenh pip de resolver PHAI giu nguyen, khong am
    # tham doi.
    import numpy as _np_hien_tai
    _pip('rembg', 'onnxruntime', 'diffusers', 'transformers', 'bitsandbytes', 'hqq',
        'sentencepiece', 'protobuf', f'numpy=={_np_hien_tai.__version__}')

    # Tai truoc vao cache HF (~10 GB) de cell 5 khong vua tai vua sinh anh lan dau. Model
    # nay KHONG bi gate (khac black-forest-labs/FLUX.1-Kontext-dev goc can chap nhan giay
    # phep + token HF) — day la ban cong dong da luong tu hoa san (DiT bnb-nf4 4bit, T5
    # hqq 4bit), tai thang duoc.
    from huggingface_hub import snapshot_download
    snapshot_download(FLUX_KONTEXT_REPO)

    print(f'>>> rembg + FLUX Kontext (4-bit) san sang ({time.time() - _t0:.0f}s). '
          f'CHUA CHAY THAT tren T4 — T4 khong co Tensor Core bf16 (kien truc Turing, sm_75) '
          f'nen phep tinh bf16 cua pipeline nay chay qua kernel thuong (cham hon Ampere+ '
          f'nhung van chay duoc); neu gap loi dtype, thu doi torch_dtype sang torch.float16 '
          f'o cell 5.')


In [ ]:
import gc, os

import torch
from PIL import Image

PREVIEW_PATH = os.path.join(WRK_DIR, 'preview_ghep_san_pham.png')
_FLUX = {'pipe': None}
_REMBG = {'session': None}


def _phien_rembg():
    """Tao 1 lan, dung lai cho cac lan goi sau (giong ban da chay thanh cong trong
    video_object_swap_colab.ipynb) — tranh nap lai model u2net moi lan."""
    if _REMBG['session'] is None:
        from rembg import new_session
        _REMBG['session'] = new_session('u2net')
    return _REMBG['session']


def tach_nen_san_pham(anh_san_pham):
    """rembg (model u2net, dung lai 1 session) -> RGBA nen TRONG SUOT, chi con san pham."""
    from rembg import remove
    im = Image.open(anh_san_pham).convert('RGB')
    return remove(im, session=_phien_rembg())


def _dan_tho_vao_khung(khung_dau_pil, sp_rgba, bbox, mo_rong=0):
    """Dan san pham (da tach nen, RGBA) vao khung_dau_pil: LAP DAY dung chieu cao cua bbox,
    tinh chieu rong theo TI LE TU NHIEN cua san pham (xem dat_san_pham_theo_ti_le o cell 3).

    XOA HET san pham CU trong CA vung bbox_mask (hop cua bbox goc + vi tri dat san pham
    moi that su) bang mau XAM TRUNG TINH (127,127,127) TRUOC khi dan san pham — day la
    quy uoc CHINH THUC cua VACE (xem docs/VACE.md cua WanGP): xam 127 = "vung nay can
    duoc SINH LAI, khong co gi o day". Neu KHONG lam buoc nay: phan bbox_mask nam NGOAI
    hinh chu nhat that su cua san pham moi (vd hai canh, khi san pham moi hep hon bbox
    san pham cu) van la pixel san pham CU tu canvas nen — day chinh la ly do san pham cu
    (vd chai) van lo ra hai ben trong anh nguoi dung gui, du mask da du rong."""
    canvas = khung_dau_pil.convert('RGBA')
    ti_le = sp_rgba.width / sp_rgba.height
    vi_tri_dat, bbox_mask = dat_san_pham_theo_ti_le(bbox, ti_le)

    mx1, my1 = int(round(bbox_mask[0])) - mo_rong, int(round(bbox_mask[1])) - mo_rong
    mx2, my2 = int(round(bbox_mask[2])) + mo_rong, int(round(bbox_mask[3])) + mo_rong
    mx1, my1 = max(0, mx1), max(0, my1)
    mx2, my2 = min(canvas.width, mx2), min(canvas.height, my2)
    xam = Image.new('RGBA', (max(1, mx2 - mx1), max(1, my2 - my1)), (127, 127, 127, 255))
    canvas.alpha_composite(xam, (mx1, my1))

    x1, y1, x2, y2 = [int(round(v)) for v in vi_tri_dat]
    nw, nh = max(1, x2 - x1), max(1, y2 - y1)
    sp_resized = sp_rgba.resize((nw, nh), Image.LANCZOS)
    canvas.alpha_composite(sp_resized, (x1, y1))
    return canvas.convert('RGB')


def _tai_flux_kontext():
    if _FLUX['pipe'] is None:
        from diffusers import FluxKontextPipeline
        _FLUX['pipe'] = FluxKontextPipeline.from_pretrained(
            FLUX_KONTEXT_REPO, torch_dtype=torch.bfloat16).to('cuda')
    return _FLUX['pipe']


def _xoa_flux_kontext():
    if _FLUX['pipe'] is not None:
        del _FLUX['pipe']
        _FLUX['pipe'] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def tao_anh_preview(video_mau, anh_san_pham, bbox_san_pham_cu, prompt_hoa_hop=None,
                    guidance_scale=None, so_buoc=None, seed=0):
    """rembg tach nen anh_san_pham -> dan tho vao khung DAU cua video_mau tai bbox ->
    (neu duoc) FLUX.1 Kontext dev hoa hop anh sang/bong do/xoa vien.

    Tach rieng 2 buoc (KHAC ban truoc gop chung 1 try/except) de biet dung buoc nao loi,
    va DE khong mat cong tach nen khi chi FLUX Kontext loi:
      - rembg loi  -> tra ve None (buoc 2 lui ve dan tho anh san pham CON NGUYEN NEN — te
        nhat, giu nguyen hanh vi cu de khong am tham hong hoan toan).
      - rembg on, FLUX Kontext loi -> tra ve anh DA TACH NEN + dan tho (chua hoa sang),
        VAN TOT HON None vi it nhat khong con nen trang/vien cung.
      - ca hai on -> tra ve anh da hoa hop day du.
    """
    if not DUNG_GHEP_ANH_PREVIEW:
        print('   [preview] bo qua (DUNG_GHEP_ANH_PREVIEW=False) -> buoc 2 tu dan tho anh san pham.')
        return None

    khung_dau = doc_khung_dau(video_mau)

    try:
        sp_rgba = tach_nen_san_pham(anh_san_pham)
    except Exception as e:
        print(f'   [preview] TACH NEN (rembg) loi ({type(e).__name__}: {str(e)[:300]}) '
              f'-> buoc 2 tu dan THO anh san pham GOC (con nguyen nen).')
        return None

    tho = _dan_tho_vao_khung(khung_dau, sp_rgba, bbox_san_pham_cu, mo_rong=MAC_DINH['mo_rong_mask'])
    duong_dan_tho = os.path.join(WRK_DIR, 'preview_da_tach_nen_chua_hoa_hop.png')
    tho.save(duong_dan_tho)
    print(f'   [preview] da tach nen + dan vao bbox -> {duong_dan_tho}')

    try:
        pipe = _tai_flux_kontext()
        gen = torch.Generator('cuda').manual_seed(seed)
        ket_qua = pipe(
            image=tho,
            prompt=prompt_hoa_hop or MAC_DINH['prompt_hoa_hop'],
            guidance_scale=guidance_scale if guidance_scale is not None else MAC_DINH['flux_guidance'],
            num_inference_steps=so_buoc or MAC_DINH['flux_steps'],
            generator=gen,
        ).images[0]
        # FluxKontextImageScale co the doi ti le/kich thuoc khung ve mot muc gan nhat trong
        # bo preset cua Flux — ep lai dung kich thuoc khung_dau de khop voi control_video.
        ket_qua = ket_qua.resize(khung_dau.size, Image.LANCZOS)
        ket_qua.save(PREVIEW_PATH)
        _moc('preview_xong')
        print(f'   [preview] FLUX Kontext hoa hop xong -> {PREVIEW_PATH}')
        return PREVIEW_PATH
    except Exception as e:
        print(f'   [preview] FLUX KONTEXT (hoa hop anh sang) loi ({type(e).__name__}: '
              f'{str(e)[:300]}) -> dung ban DA TACH NEN nhung CHUA hoa hop lam anh preview '
              f'({duong_dan_tho}).')
        return duong_dan_tho
    finally:
        _xoa_flux_kontext()


In [ ]:
import gc, os, subprocess, sys, time

import numpy as np
import torch
from PIL import Image

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path.insert(0, COMFY)
os.chdir(COMFY)   # ComfyUI tim models/ input/ theo cwd

from nodes import CLIPLoader, CLIPTextEncode, VAEDecode, VAELoader, KSampler, LoraLoaderModelOnly
from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_wan import WanVaceToVideo, TrimVideoLatent
from comfy import model_management


def _don():
    try:
        model_management.unload_all_models()
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def _tensor_sang_u8(decoded):
    """(N,H,W,3) float [0,1] -> uint8. NaN/rong (moi pixel nhu nhau) la dau hieu tran so
    hoac latent hong — bao loi NGAY thay vi ghi video den ma job van 'coi nhu xong'."""
    x = decoded.detach().float().cpu()
    if not torch.isfinite(x).all():
        raise RuntimeError('tensor giai ma co NaN/Inf — pipeline tran so (fp16 tren T4?)')
    x = (x.clamp(0, 1) * 255).round().to(torch.uint8).numpy()
    if x.std() < 0.5:
        raise RuntimeError(f'video ra RONG (do lech chuan {x.std():.3f}/255) — moi pixel nhu nhau')
    return x


def _anh_sang_tensor(pil_img, w, h):
    """PIL RGB -> tensor (1,H,W,3) float [0,1], resize LANCZOS ve dung (w,h)."""
    im = pil_img.convert('RGB').resize((w, h), Image.LANCZOS)
    return torch.from_numpy(np.asarray(im)).float().unsqueeze(0) / 255.0


def _anh_composite_toan_khung(rong, cao, bbox_scaled, khung0_pil, anh_preview, anh_san_pham, mo_rong=0):
    """Tra ve MOT anh kich thuoc (rong,cao) — toan bo khung canh nhu video mau, nhung
    vung bbox da co san SAN PHAM MOI. Day la thu se duoc dan vao moi khung cua
    control_video (xem doi_san_pham_video) — KHONG the dua thang anh_san_pham (anh
    studio nen trang, khong cung ti le/vi tri voi bbox) vao do duoc.

    - Co anh_preview (buoc 1: rembg + FLUX Kontext, xem cell 5): no da la toan canh +
      san pham dat dung cho, da hoa hop anh sang/bong do, da XOA san pham cu bang mau
      xam (xem _dan_tho_vao_khung o cell 5) -> chi resize ve (rong,cao).
    - Khong co (buoc 1 tat/loi): tu XOA san pham CU trong CA vung bbox_mask bang mau XAM
      TRUNG TINH (127,127,127 — quy uoc chinh thuc cua VACE, xem docs/VACE.md cua WanGP:
      "grey = vung can duoc sinh lai"), roi dan san pham moi (LAP DAY chieu cao bbox,
      tinh chieu rong theo dung ti le tu nhien — xem dat_san_pham_theo_ti_le o cell 3)
      len tren. KHONG xoa xam truoc se de lo pixel san pham CU o phan bbox_mask nam
      NGOAI hinh chu nhat that su cua san pham moi (vd hai canh, khi san pham moi hep
      hon bbox san pham cu) — day chinh la ly do san pham cu (vd chai) van lo ra hai ben
      du mask da du rong (xem CHANGELOG v7)."""
    if anh_preview:
        return doc_anh_rgb(anh_preview).resize((rong, cao), Image.LANCZOS)

    canvas = khung0_pil.convert('RGB').resize((rong, cao), Image.LANCZOS)
    sp = doc_anh_rgb(anh_san_pham)
    ti_le_sp = sp.width / sp.height
    vi_tri_dat, bbox_mask = dat_san_pham_theo_ti_le(bbox_scaled, ti_le_sp)

    mx1 = max(0, int(round(bbox_mask[0])) - mo_rong)
    my1 = max(0, int(round(bbox_mask[1])) - mo_rong)
    mx2 = min(rong, int(round(bbox_mask[2])) + mo_rong)
    my2 = min(cao, int(round(bbox_mask[3])) + mo_rong)
    canvas_arr = np.array(canvas)
    canvas_arr[my1:my2, mx1:mx2] = 127
    canvas = Image.fromarray(canvas_arr)

    x1, y1, x2, y2 = [int(round(v)) for v in vi_tri_dat]
    nw, nh = max(1, x2 - x1), max(1, y2 - y1)
    sp_resized = sp.resize((nw, nh), Image.LANCZOS)
    canvas.paste(sp_resized, (x1, y1))
    return canvas


def _rife(src, dst, boi, fps_ra):
    """Noi suy khung x`boi` bang Practical-RIFE. Hong thi lui ve ffmpeg minterpolate."""
    try:
        subprocess.run([sys.executable, 'inference_video.py', f'--multi={boi}', f'--fps={fps_ra}',
                        f'--video={src}', '--scale=1', f'--output={dst}'],
                       cwd=RIFE, check=True, capture_output=True, text=True, timeout=1800)
        if os.path.exists(dst) and os.path.getsize(dst) > 1000:
            return 'rife'
        raise RuntimeError('RIFE khong ghi file')
    except Exception as e:
        print(f'   [{BACKEND}] RIFE hong ({type(e).__name__}: {str(e)[:200]}) -> dung ffmpeg minterpolate')
        subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', src,
                        '-vf', f"minterpolate='fps={fps_ra}:mi_mode=mci:mc_mode=aobmc:vsbmc=1'",
                        '-c:v', 'libx264', '-crf', '18', '-pix_fmt', 'yuv420p', dst], check=True)
        return 'minterpolate'


def doi_san_pham_video(video_mau, anh_san_pham, bbox_san_pham_cu, out_path, anh_preview=None,
                       mo_ta_san_pham_moi=None, negative_prompt=None, so_frame=None,
                       moi_khung_thu=None, khung_moi_giay=None, rong=None, cao=None, steps=None,
                       guidance=None, seed=42, fps_ra=None, mo_rong_mask=None,
                       dung_lora_toc_do=None, progress=None):
    """Sinh video moi: giu NGUYEN chuyen dong/nen/camera cua `video_mau`, doi san pham
    trong vung `bbox_san_pham_cu` (pixel cua KHUNG HINH GOC cua video_mau) sang san pham
    trong `anh_san_pham` (hoac `anh_preview` da ghep san o buoc 1 neu co).

    Co che THAT (doc tu chinh source comfy_extras/nodes_wan.py, ham WanVaceToVideo.encode
    — v2 dua theo mo ta gian luoc cua tai lieu ComfyUI nen SAI, xem CHANGELOG v3):
        inactive = control_video * (1 - mask)      # vung mask=0: giu NGUYEN control_video
        reactive = control_video * mask            # vung mask=1: CUNG la control_video!
    Ca hai nhanh deu lay pixel TU CHINH control_video — mask chi quyet dinh nhanh nao
    thay pixel that, nhanh kia thay xam (0.5). KHONG co chuyen "vung mask=1 tu dong duoc
    ve lai theo reference_image" — reference_image chi la MOT KHUNG THAM CHIEU rieng,
    noi VAO DAU chuoi latent (mot dang "goi y" toan cuc yeu), khong ep pixel vao vung
    mask. Ket qua v2 (do that): output = gan nhu NGUYEN VIDEO GOC, khong thay san pham
    moi dau ca — vi (a) guidance=1.0 (bat buoc de dung LoRA lightx2v 4-buoc) lam prompt
    mo ta san pham moi gan nhu vo tac dung, va (b) vung "reactive" van la ANH CHAI NUOC
    HOA CU (tu control_video), nen model chi hoc duoc "ve lai dung cai da co".

    Sua o v3: TU DAN san pham moi vao MOI khung cua control_video (ham
    `_anh_composite_toan_khung`) tai vi tri bbox, TRUOC KHI dua vao WanVaceToVideo. Nho
    the, o vung mask=1, `reactive = control_video_da_dan * mask` gio la SAN PHAM MOI chu
    khong phai san pham cu — model chi con phai "lam that/hoa hop" (anh sang, chuyen
    dong nuoc) thay vi phai "bia ra tu con so" duoi guidance=1.0. Ngoai bbox (mask=0)
    van la pixel that cua video_mau — nen/nuoc/camera khong doi.

    CHUA DO LAI tren T4 sau sua v3 — chi la suy luan dung tu doc source, chua kiem
    chung bang mat cho dung video nay.
    """
    LAST_STATS.clear()
    t0 = time.time()
    uid = os.path.splitext(os.path.basename(out_path))[0].split('_')[0] or 'job'

    mo_ta_san_pham_moi = mo_ta_san_pham_moi or MAC_DINH['mo_ta_san_pham_moi']
    negative_prompt = negative_prompt or MAC_DINH['negative_prompt']
    so_frame = so_frame or MAC_DINH['so_frame']
    khung_moi_giay = float(khung_moi_giay or MAC_DINH['khung_moi_giay'])
    steps = steps or MAC_DINH['steps']
    guidance = MAC_DINH['guidance'] if guidance is None else guidance
    fps_ra = fps_ra or MAC_DINH['fps_ra']
    mo_rong_mask = MAC_DINH['mo_rong_mask'] if mo_rong_mask is None else mo_rong_mask
    dung_lora_toc_do = MAC_DINH['dung_lora_toc_do'] if dung_lora_toc_do is None else dung_lora_toc_do

    vw, vh, vfps, vn = probe_video(video_mau)
    if rong is None or cao is None:
        rong, cao = _kich_thuoc_theo_anh(vw, vh, MAC_DINH['ngan_sach_pixel'], boi=16)
    moi_thu = int(moi_khung_thu) if moi_khung_thu else max(1, int(round(vfps / khung_moi_giay)))
    co_that = (vn - 1) // moi_thu + 1 if vn else 10 ** 9
    n = _khung_4n1(min(int(so_frame), co_that))
    if n < so_frame:
        print(f'   [{BACKEND}] so_frame {so_frame} -> {n} (video mau chi co {co_that} khung '
              f'khi lay moi khung thu {moi_thu}, va can 4n+1)')
    fps_sinh = vfps / moi_thu
    boi = max(1, int(round(fps_ra / fps_sinh)))

    bbox_scaled = doi_ti_le_bbox(bbox_san_pham_cu, vw, vh, rong, cao)
    LAST_STATS.update({'_case': TEST_CASE, 'video_mau': [vw, vh, vfps, vn], 'video_ra': [rong, cao, n, fps_ra],
                       'moi_khung_thu': moi_thu, 'fps_sinh': round(fps_sinh, 2), 'rife_x': boi,
                       'steps': steps, 'cfg': guidance, 'seed': seed, 'quant': QUANT_VACE,
                       'dung_lora_toc_do': dung_lora_toc_do,
                       'bbox_scaled': [round(v, 1) for v in bbox_scaled], 'dung_preview': anh_preview is not None})
    print(f'   [{BACKEND}] video mau {vw}x{vh}@{vfps:.0f}fps x{vn} -> lay moi khung thu {moi_thu} '
          f'({fps_sinh:.1f} fps), sinh {n} khung {rong}x{cao}, RIFE x{boi} -> {fps_ra} fps')

    with torch.inference_mode():
        if progress: progress('ma_hoa_prompt', 0, 8)
        clip = CLIPLoader().load_clip(os.path.basename(TE), 'wan', 'default')[0]
        pos = CLIPTextEncode().encode(clip, mo_ta_san_pham_moi)[0]
        neg = CLIPTextEncode().encode(clip, negative_prompt)[0]
        del clip
        _don()

        if progress: progress('doc_video_mau', 1, 8)
        khung, _ = doc_khung_video(video_mau, moi_thu, n)   # (n,vh,vw,3) [0,1]
        khung = torch.nn.functional.interpolate(
            khung.permute(0, 3, 1, 2), size=(cao, rong), mode='bilinear', align_corners=False
        ).permute(0, 2, 3, 1).clamp(0, 1)

        khung0_pil = doc_khung_dau(video_mau)
        anh_composite_pil = _anh_composite_toan_khung(rong, cao, bbox_scaled, khung0_pil,
                                                       anh_preview, anh_san_pham, mo_rong=mo_rong_mask)
        anh_ref = _anh_sang_tensor(anh_composite_pil, rong, cao)   # (1,cao,rong,3)

        if progress: progress('tao_mask', 2, 8)
        # bbox_mask = HOP cua bbox_scaled (vung san pham CU) va vi_tri_dat (vung san pham
        # MOI, lap day chieu cao bbox theo dung ti le tu nhien — xem
        # dat_san_pham_theo_ti_le o cell 3). Neu chi dung bbox_scaled tho: san pham moi co
        # the hep hon/rong hon bbox nguoi dung khoanh, de lo pixel san pham CU o phan
        # khong duoc mask che (chinh la loi nguoi dung gap: chai cu khong bi xoa het).
        _anh_sp_goc = doc_anh_rgb(anh_san_pham)
        ti_le_sp_that = _anh_sp_goc.width / _anh_sp_goc.height
        _, bbox_mask = dat_san_pham_theo_ti_le(bbox_scaled, ti_le_sp_that)
        mask_1 = mat_na_tu_bbox(rong, cao, bbox_mask, mo_rong=mo_rong_mask)   # (cao,rong)
        mask_bool = torch.from_numpy(mask_1 > 0.5)
        # Dan san pham MOI (tu anh_ref) vao dung vi tri mask cua MOI khung control_video —
        # ly do bat buoc, xem docstring o tren (WanVaceToVideo lay ca 'reactive' TU
        # control_video, khong tu reference_image).
        khung[:, mask_bool] = anh_ref[0, mask_bool]
        control_masks = torch.from_numpy(mask_1).float().unsqueeze(0).repeat(n, 1, 1)   # (n,cao,rong)

        if progress: progress('ma_hoa_vace', 3, 8)
        vae = VAELoader().load_vae(os.path.basename(VAE))[0]
        pos_o, neg_o, latent, trim = WanVaceToVideo().encode(
            pos, neg, vae, rong, cao, n, 1, 1.0, khung, control_masks, anh_ref)
        del khung
        _don()

        if progress: progress('nap_model', 4, 8)
        model = UnetLoaderGGUF().load_unet(os.path.basename(DIT))[0]
        if dung_lora_toc_do:
            model = LoraLoaderModelOnly().load_lora_model_only(
                model, os.path.basename(LORA_LIGHTX2V), MAC_DINH['lightx2v'])[0]
        model = ModelSamplingSD3().patch(model, MAC_DINH['shift'])[0]

        if progress: progress('denoise', 5, 8)
        sampled = KSampler().sample(model=model, seed=seed, steps=steps, cfg=guidance,
                                    sampler_name=MAC_DINH['sampler'], scheduler=MAC_DINH['scheduler'],
                                    positive=pos_o, negative=neg_o, latent_image=latent)[0]
        del model
        _don()
        sampled = TrimVideoLatent().op(sampled, trim)[0]

        if progress: progress('giai_ma', 6, 8)
        decoded = VAEDecode().decode(vae, sampled)[0]
        del vae, sampled
        _don()

    if progress: progress('ghi_video', 7, 8)
    u8 = _tensor_sang_u8(decoded)
    del decoded
    tho = os.path.join(WRK_DIR, f'{uid}_tho.mp4')
    encode_video(save_frames_u8(u8, os.path.join(WRK_DIR, uid)), fps_sinh, tho, tag='tho')
    ns = os.path.join(WRK_DIR, f'{uid}_rife.mp4')
    LAST_STATS['noi_suy'] = _rife(tho, ns, boi, fps_ra)
    reencode_video(ns, out_path, tag=f'case={_slug(TEST_CASE)} seed={seed}')
    _don()
    LAST_STATS['_thoi_gian_s'] = round(time.time() - t0, 1)
    _moc('xong')
    print(f'   [{BACKEND}] xong {n} khung (+RIFE) trong {time.time() - t0:.0f}s -> {out_path}')
    return out_path


In [ ]:
import os

import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Video, display

# ============================================================================
# THAM SO CHAY THU — sua 3 dong nay cho ca cua ban. Vi du duoi dang de KHOP voi
# 3 file co san trong thu muc (3d-waterfall-billboard.mp4 + photo_..._x.jpg).
# ============================================================================
VIDEO_MAU = '/content/video_mau.mp4'          #@param {type:"string"}
ANH_SAN_PHAM = '/content/anh_san_pham.jpg'    #@param {type:"string"}


def _dam_bao_co_file(path, goi_y):
    if os.path.exists(path):
        return path
    print(f'   khong thay {path} -> mo hop thoai upload cho "{goi_y}" (Colab: chon 1 file).')
    from google.colab import files
    up = files.upload()
    ten = next(iter(up))
    if os.path.abspath(ten) != os.path.abspath(path):
        os.replace(ten, path)
    return path


VIDEO_MAU = _dam_bao_co_file(VIDEO_MAU, 'video mau (co san pham CU)')
ANH_SAN_PHAM = _dam_bao_co_file(ANH_SAN_PHAM, 'anh san pham MOI')

khung_dau = doc_khung_dau(VIDEO_MAU)
print(f'kich thuoc khung hinh dau cua video mau: {khung_dau.size} (rong x cao)')
plt.figure(figsize=(6, 6 * khung_dau.height / khung_dau.width))
plt.imshow(khung_dau)
plt.xticks(range(0, khung_dau.width, 50), rotation=90)
plt.yticks(range(0, khung_dau.height, 50))
plt.grid(color='yellow', alpha=0.4, linewidth=0.5)
plt.title('Doc toa do (x,y) vung SAN PHAM CU tren luoi nay -> dien vao BBOX_SAN_PHAM_CU')
plt.show()

# BBOX_SAN_PHAM_CU: (x1,y1,x2,y2) pixel cua khung hinh GOC (nhu anh luoi o tren), goc
# tren-trai la (0,0). VI DU duoi la UOC LUONG BANG MAT cho 3d-waterfall-billboard.mp4
# (720x1280) — TU KIEM TRA LAI bang anh luoi phia tren truoc khi chay that, chua do
# chinh xac tung pixel.
BBOX_SAN_PHAM_CU = (200, 125, 475, 875)    #@param {type:"raw"}
MO_TA_SAN_PHAM_MOI = (
    'a pink oversized button-up shirt hovering in place of the perfume bottle, '
    'exact same billboard advertisement scene, same glass skyscraper, same cascading '
    'waterfall water effect pouring down, same camera angle and lighting, '
    'photorealistic, high quality'
)    #@param {type:"string"}

print('Vung se bi thay the (kiem tra khop voi san pham CU truoc khi chay tiep):')
display(ve_bbox_len_anh(khung_dau, BBOX_SAN_PHAM_CU))

# ---- Buoc 1 (tuy chon): ghep thu -> anh xem truoc ----
anh_preview = tao_anh_preview(VIDEO_MAU, ANH_SAN_PHAM, BBOX_SAN_PHAM_CU)
if anh_preview:
    print('Anh xem truoc (rembg tach nen + FLUX Kontext hoa hop):')
    display(Image.open(anh_preview))
    print('Ung y thi chay tiep buoc 2. Khong ung y: doi ANH_SAN_PHAM / BBOX_SAN_PHAM_CU roi chay lai '
         'CHI cell nay (khong can chay lai cell cai dat).')
else:
    print('Khong co anh xem truoc -> buoc 2 se dung THANG anh san pham goc lam reference_image.')

# ---- Buoc 2: sinh video (v3 do that 548s VOI LoRA/4-buoc; mac dinh v4 KHONG con LoRA
# + 25 buoc nen se CHAM HON NHIEU — CHUA DO LAI, xem CHANGELOG cell 1) ----
OUT_PATH = os.path.join(OUT_DIR, 'video_da_doi_san_pham.mp4')
doi_san_pham_video(VIDEO_MAU, ANH_SAN_PHAM, BBOX_SAN_PHAM_CU, OUT_PATH,
                   anh_preview=anh_preview, mo_ta_san_pham_moi=MO_TA_SAN_PHAM_MOI)

print('LAST_STATS:', LAST_STATS)
display(Video(OUT_PATH, embed=True, width=360))


## Ghi chú vận hành / troubleshooting

- **Camera trong video mẫu di chuyển/zoom**: `BBOX_SAN_PHAM_CU` hiện là MỘT khung cố
  định lặp lại cho mọi frame. Nếu camera đứng yên (như video mẫu đi kèm) thì ổn; nếu
  camera di chuyển, cần bbox theo từng khung (cải tiến: theo dõi bbox bằng
  [SAM2](https://github.com/facebookresearch/sam2) từ một cú click ở khung đầu, thay
  vì một bbox tĩnh).
- **`video ra RONG` / NaN**: dấu hiệu tràn số fp16 trên T4 (bài học từ notebook VACE
  kia) — thử hạ `QUANT_VACE` xuống `Q4_K_M`, hoặc giảm `so_frame`.
- **Sản phẩm mới bị "méo" theo hình dạng sản phẩm cũ**: tăng `mo_rong_mask` (mặc định
  24px) để mask rộng hơn quanh vị trí sản phẩm cũ, cho VACE nhiều chỗ hơn để vẽ hình
  dạng khác (ví dụ chai tròn → áo sơ mi vuông).
- **Muốn chất lượng ghép ảnh preview tốt hơn FLUX Kontext 4-bit**: có thể thay bước 1
  bằng gọi API Gemini 2.5 Flash Image ("Nano Banana") — đây gần như chắc chắn là model
  các web mẫu (Higgsfield-kiểu "product replace") dùng thật, chất lượng cao hơn hẳn
  bất kỳ model local nào, không tốn VRAM T4, nhưng cần `GEMINI_API_KEY` và tốn phí API
  nhỏ mỗi lần. Chỉ cần thay nội dung hàm `tao_anh_preview()` ở cell 5, phần còn lại của
  pipeline không đổi (`anh_preview` vẫn chỉ là một đường dẫn PNG).
- **Job quá lâu với 25 bước/không LoRA (v4)**: đặt `MAC_DINH['dung_lora_toc_do'] = True`
  ở cell 1 rồi chạy lại (không cần chạy lại cell cài đặt) để quay về tốc độ nhanh của
  v3 (4 bước, cfg=1) — đổi lại vùng sản phẩm mới sẽ ít được "làm thật hoá" hơn.
- **FLUX Kontext lỗi cài đặt/CUDA**: đặt `DUNG_GHEP_ANH_PREVIEW = False` ở cell 1 — bước
  2 (VACE) hoàn toàn độc lập, tự dán thô ảnh sản phẩm gốc thay vì dùng ảnh đã hoà hợp.
- **`AI_TYPE`/`BACKEND_PARAMS`** ở cell 1 để SẴN theo đúng khuôn `/generate/<type>` của
  hai notebook kia — notebook này KHÔNG dựng server FastAPI/callback/cloudflared (chạy
  tay, theo lựa chọn khi tạo notebook). Muốn cắm vào "cụm Colab" thì bê logic
  `doi_san_pham_video()` + `tao_anh_preview()` sang khung server của
  `ImageVideoToVideo_WanVACE_Colab.ipynb` (cell cuối của notebook đó).

## Hướng cải tiến tiếp theo (chưa làm ở v1)

1. Theo dõi mask qua nhiều khung bằng SAM2 thay vì bbox tĩnh (camera động).
2. Cache prompt embeds theo `(mo_ta_san_pham_moi, negative_prompt)` như notebook VACE
   kia đã làm (bớt ~30s/lần chạy lại với cùng prompt).
3. Đo thật thời gian/VRAM từng bước trên T4 và ghi vào `CHANGELOG` — v1 hiện tại là suy
   luận từ tài liệu + số đo của notebook I2V/VACE khác, CHƯA phải số đo của chính
   notebook này.
